In [ ]:
import pandas as pd

import biogeme.biogeme as bio
from biogeme import models
from biogeme.database import Database
from biogeme.expressions import Beta, Variable

In [ ]:
df = pd.read_excel('RVM_data_project_clean.xlsx')
df['Age_10'] = df['Age'] / 10
database = Database('RVM_data_project_clean.xlsx', df)
df.head()

In [ ]:
choice_code = Variable('choice_code')

Age_10 = Variable('Age_10')
Female = Variable('Female')

dist = {i: Variable(f'dist_{i}') for i in range(1, 36)}
indoor = {i: Variable(f'indoor_{i}') for i in range(1, 36)}

shopping_center = {i: Variable(f'shopping_center_{i}') for i in range(1, 36)}
university = {i: Variable(f'university_{i}') for i in range(1, 36)}
public_building = {i: Variable(f'public_building_{i}') for i in range(1, 36)}
private_building = {i: Variable(f'private_building_{i}') for i in range(1, 36)}
public_park = {i: Variable(f'public_park_{i}') for i in range(1, 36)}
hospital = {i: Variable(f'hospital_{i}') for i in range(1, 36)}
technical_institution = {i: Variable(f'technical_institution_{i}') for i in range(1, 36)}
science_park = {i: Variable(f'science_park_{i}') for i in range(1, 36)}
recreational_park = {i: Variable(f'recreational_park_{i}') for i in range(1, 36)}
downtown_recycling = {i: Variable(f'downtown_recycling_{i}') for i in range(1, 36)}
conv_center = {i: Variable(f'conv_center_{i}') for i in range(1, 36)}
industrial_business_center = {i: Variable(f'industrial_business_center_{i}') for i in range(1, 36)}

Model 9 tutor advice
1. Exclude machines belonging to location categories with 0 observed choices:
   - `downtown_recycling`
   - `recreational_park`
   - `hospital`
   - `public_park`
2. Use distance, indoor, age and female 
3. Add ASCs

In [ ]:
zero_location_vars = [
    'downtown_recycling',
    'recreational_park',
    'hospital',
    'public_park',
]
machines_to_keep = []

for i in range(1,36):
    is_zero_location = (
        df[f'downtown_recycling_{i}'].max()
        + df[f'recreational_park_{i}'].max()
        + df[f'hospital_{i}'].max()
        + df[f'public_park_{i}'].max()
    )

    if is_zero_location == 0:
        machines_to_keep.append(i)

print('Machines:', machines_to_keep)

In [ ]:
building = {}

for i in machines_to_keep:
    building[i] = public_building[i] + private_building[i]

In [ ]:
B_DIST = Beta('B_DIST', 0, None, None, 0)
B_INDOOR = Beta('B_INDOOR', 0, None, None, 0)
B_AGE = Beta('B_AGE', 0, None, None, 0)
B_FEMALE = Beta('B_FEMALE', 0, None, None, 0)
B_BUILDING = Beta('B_BUILDING', 0, None, None, 0)

In [ ]:
ASC = {}
reference_alt = machines_to_keep[0]

for i in machines_to_keep:
    if i == reference_alt:
        ASC[i] = Beta(f'ASC_{i}', 0, None, None, 1)  # fixed to zero according to Train in his book 
    else:
        ASC[i] = Beta(f'ASC_{i}', 0, None, None, 0)

In [ ]:
V = {}

for i in machines_to_keep:
    V[i] = (
        ASC[i]
        + B_DIST * dist[i]
        + B_INDOOR * indoor[i]
        + B_AGE * Age_10
        + B_FEMALE * Female
        + B_BUILDING * building[i]
    )

av = {i: 1 for i in machines_to_keep}

logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model9_tutor_advice'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

Model 10:
1. Uses machines with more than 0 observations.
2. Groups the remaining location types.
3. Uses `commercial` as the omitted baseline group.
4. Adds ASCs.

Groups:
- `commercial = shopping_center + conv_center`  baseline
- `campus = university + technical_institution + science_park`
- `building = public_building + private_building`
- `business = industrial_business_center`


In [ ]:
commercial = {}
campus = {}
building = {}
business = {}

for i in machines_to_keep:
    commercial[i] = (
        shopping_center[i] 
        + conv_center[i]
    )
    campus[i] = (
        university[i]
        + technical_institution[i]
        + science_park[i]
    )

    building[i] = (
        public_building[i]
        + private_building[i]
    )

    business[i] = industrial_business_center[i]

In [ ]:
B_DIST = Beta('B_DIST', 0, None, None, 0)
B_INDOOR = Beta('B_INDOOR', 0, None, None, 0)
B_DIST_INDOOR = Beta('B_DIST_INDOOR', 0, None, None, 0)

B_INDOOR_AGE = Beta('B_INDOOR_AGE', 0, None, None, 0)
B_INDOOR_FEMALE = Beta('B_INDOOR_FEMALE', 0, None, None, 0)

B_CAMPUS = Beta('B_CAMPUS', 0, None, None, 0)
B_BUILDING = Beta('B_BUILDING', 0, None, None, 0)
B_BUSINESS = Beta('B_BUSINESS', 0, None, None, 0)

In [ ]:
ASC = {}
reference_alt = machines_to_keep[0]

for i in machines_to_keep:
    if i == reference_alt:
        ASC[i] = Beta(f'ASC_{i}', 0, None, None, 1)  
    else:
        ASC[i] = Beta(f'ASC_{i}', 0, None, None, 0)

In [ ]:
V = {}

for i in machines_to_keep:
    V[i] = (
        ASC[i]
        + B_DIST * dist[i]
        + B_INDOOR * indoor[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
        + B_INDOOR_AGE * indoor[i] * Age_10
        + B_INDOOR_FEMALE * indoor[i] * Female
        + B_CAMPUS * campus[i]
        + B_BUILDING * building[i]
        + B_BUSINESS * business[i]
    )

av = {i: 1 for i in machines_to_keep}
logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model10_tutor_advice'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

Model 11: same as Model 10, but without ASCs

In [ ]:
V = {}

for i in machines_to_keep:
    V[i] = (
        B_DIST * dist[i]
        + B_INDOOR * indoor[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
        + B_INDOOR_AGE * indoor[i] * Age_10
        + B_INDOOR_FEMALE * indoor[i] * Female
        + B_CAMPUS * campus[i]
        + B_BUILDING * building[i]
        + B_BUSINESS * business[i]
    )

av = {i: 1 for i in machines_to_keep}
logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model11_tutor_advice'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

Model 12: Because we see that in model 11, CAMPUS is not that significant we are removing it, we will also add dist_age

In [ ]:
V = {}
B_DIST_AGE = Beta('B_DIST_AGE', 0, None, None, 0)

for i in machines_to_keep:
    V[i] = (
        B_DIST * dist[i]
        + B_DIST_AGE * dist[i] * Age_10
        + B_INDOOR * indoor[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
        + B_INDOOR_AGE * indoor[i] * Age_10
        + B_INDOOR_FEMALE * indoor[i] * Female
        + B_BUILDING * building[i]
        + B_BUSINESS * business[i]
    )

av = {i: 1 for i in machines_to_keep}
logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model12_tutor_advice'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

Last model: adding opt_out

In [ ]:
V = {}
B_DIST_AGE = Beta('B_DIST_AGE', 0, None, None, 0)
OUT = 36
ASC_OUT = Beta('ASC_OUT', 0, None, None, 1)
for i in machines_to_keep:
    V[i] = (
        B_DIST * dist[i]
        #+ B_DIST_AGE * dist[i] * Age_10
        + B_INDOOR * indoor[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
        + B_INDOOR_AGE * indoor[i] * Age_10
        + B_INDOOR_FEMALE * indoor[i] * Female
        + B_BUILDING * building[i]
        + B_BUSINESS * business[i]
    )
V[OUT] = ASC_OUT
av = {i: 1 for i in machines_to_keep}
av[OUT] = 1
logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model_last_no_distage'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())

one last check with all 35 machines

In [ ]:
machines_to_keep = list(range(1, 36))
OUT = 36



V = {}
B_DIST_AGE = Beta('B_DIST_AGE', 0, None, None, 0)
ASC_OUT = Beta('ASC_OUT', 0, None, None, 0)

for i in machines_to_keep:
    V[i] = (
        B_DIST * dist[i]
        #+ B_DIST_AGE * dist[i] * Age_10
        + B_INDOOR * indoor[i]
        + B_DIST_INDOOR * dist[i] * indoor[i]
        + B_INDOOR_AGE * indoor[i] * Age_10
        + B_INDOOR_FEMALE * indoor[i] * Female
        + B_BUILDING * building[i]
        + B_BUSINESS * business[i]
    )
V[OUT] = ASC_OUT

av = {i: 1 for i in machines_to_keep}
av[OUT] = 1
logprob = models.loglogit(V, av, choice_code)

biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'rvm_model_last_no_distage_all_machines'

results = biogeme.estimate()

print(results.short_summary())
print(results.get_estimated_parameters())